In [12]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

import requests

In [2]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
question = 'what is the capital of france'
a = llm(question)

print(a)

JSON data

In [21]:
docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [22]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [23]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [25]:
documents[50]

{'id': 'd677df9ccb',
 'course': 'data-engineering-zoomcamp',
 'section': 'Module 1: Taxi Data (download & handling)',
 'question': 'Taxi Data: How to handle *.csv.gz taxi data files?',
 'answer': 'In [this video](https://www.youtube.com/watch?v=B1WwATwf-vY&list=PL3MmuxUbc_hJed7dXYoJw8DoCuVHhGEQb), the data file is stored as `output.csv`. If the file extension is `csv.gz` instead of `csv`, it won\'t store correctly.\n\nTo handle this:\n\n1. Replace `csv_name = "output.csv"` with the file name extracted from the URL. For example, for the yellow taxi data, use:\n   \n   ```python\n   url = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2021-01.csv.gz"\n   csv_name = url.split("/")[-1]\n   ```\n\n2. When you use `csv_name` with `pandas.read_csv`, it will work correctly because `pandas.read_csv` can directly read files with the `csv.gz` extension.\n\nExample:\n\n```python\nimport pandas as pd\n\nurl = "https://github.com/DataTalksClub/nyc-tlc-data/re

Search

In [36]:
from minsearch import Index

#"question", "section", "answer" - are the field we are looking at
#course - this is the exact match keyword



index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
search_results = index.search(question, filter_dict={'course':'llm-zoomcamp'}, num_results=5)

Let's wrap the search in a search function - the first component of our RAG pipeline:


In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}   # below 1 if word appear in field this is less important than over 1
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [10]:
prompt = f'''
Your task is to answer question from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers.
If the answer is not found in the contxt, respond with "I dont know"

Question: 
{question}

Context:
{context}

'''


NameError: name 'context' is not defined